# 📊 Unemployment in India — Professional Data Analysis & Multi-Model Prediction
---
| | |
|---|---|
| **Author** | Data Science Lab |
| **Dataset 1** | `Unemployment_in_India.csv` — 768 records · 28 states/UTs · Rural & Urban breakdown |
| **Dataset 2** | `Unemployment_Rate_upto_11_2020.csv` — 267 records · Jan–Nov 2020 · with geo-coordinates |
| **Period** | May 2019 – Nov 2020 (includes COVID-19 lockdown shock) |
| **Target** | `Estimated Unemployment Rate (%)` |
| **Models** | 10 algorithms — Linear → Tree-based → Gradient Boosting → XGBoost → LightGBM |

---
## Table of Contents
1. [Library Imports & Global Settings](#1)
2. [Data Loading & Cleaning](#2)
3. [Dataset Overview & Missing Value Analysis](#3)
4. [National Unemployment Trend Over Time](#4)
5. [Rural vs Urban Unemployment Over Time](#5)
6. [COVID-19 Impact Analysis](#6)
7. [State-wise Unemployment — Bar & Box Plots](#7)
8. [Zone-wise Distribution (Dataset 2)](#8)
9. [Scatter Analysis — Feature Relationships](#9)
10. [Distribution Analysis — Histograms & KDE](#10)
11. [Violin & Strip Overlay Plots](#11)
12. [Pair Plot — All Feature Combinations](#12)
13. [Correlation Heatmap & Region×Month Heatmap](#13)
14. [Feature Engineering](#14)
15. [Train / Test Split & Scaling](#15)
16. [Training 10 Models — Cross-Validated Comparison](#16)
17. [Model Performance Visualisation](#17)
18. [Best Model — Hyperparameter Tuning (GridSearchCV)](#18)
19. [Final Model Evaluation on Test Set](#19)
20. [Actual vs Predicted & Residual Analysis](#20)
21. [Feature Importance & SHAP-style Analysis](#21)
22. [Learning Curves](#22)
23. [Prediction Error Distribution](#23)
24. [Comprehensive Model Summary Table](#24)
25. [Conclusions](#25)

---
## 1. Library Imports & Global Settings <a id='1'></a>

All libraries are imported in one place. We configure a consistent, publication-quality visual theme across all plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = 'Set2'
ACCENT  = '#2563EB'
SEED    = 42


---
## 2. Data Loading & Cleaning <a id='2'></a>

Both datasets are loaded, column names are normalised (whitespace stripped), the `Date` column is parsed to `datetime`, and columns are renamed to clean, snake_case identifiers. A Frequency column inconsistency (`'Monthly'` vs `' Monthly'`) is also fixed.

In [ ]:
df  = pd.read_csv('Unemployment_in_India.csv')
df2 = pd.read_csv('Unemployment_Rate_upto_11_2020.csv')

for frame in [df, df2]:
    frame.columns = frame.columns.str.strip()

RENAME = {
    'Estimated Unemployment Rate (%)':      'Unemployment_Rate',
    'Estimated Employed':                   'Employed',
    'Estimated Labour Participation Rate (%)': 'LPR',
}
df.rename(columns=RENAME, inplace=True)
df2.rename(columns={**RENAME, 'Region.1': 'Zone'}, inplace=True)

df['Date']  = pd.to_datetime(df['Date'].str.strip(),  dayfirst=True, errors='coerce')
df2['Date'] = pd.to_datetime(df2['Date'].str.strip(), dayfirst=True, errors='coerce')

df['Frequency']  = df['Frequency'].str.strip()
df2['Frequency'] = df2['Frequency'].str.strip()

df['Month']  = df['Date'].dt.month
df['Year']   = df['Date'].dt.year
df['Quarter']= df['Date'].dt.quarter
df['YM']     = df['Date'].dt.to_period('M').astype(str)

print(f'Dataset 1 shape  : {df.shape}')
print(f'Dataset 2 shape  : {df2.shape}')
print(f'Date range (df)  : {df["Date"].min().date()}  →  {df["Date"].max().date()}')
print(f'Date range (df2) : {df2["Date"].min().date()} →  {df2["Date"].max().date()}')
df.head(10)